# Participación regional de la demanda industrial (códigos OSeMOSYS, eficiencias agrupadas)

Genera `Insumos/Participacion_Regional_Industrial.xlsx` a partir de `Insumos/Datos_Industrial_Regionalizado.xlsx` (una hoja por región), **equivalente al proceso residencial corregido**: ya no se calcula la participación por `Uso\fuel\eficiencia` sino **solo por `Uso\fuel`** — se suman las tres eficiencias (`Eficiencia_existente`, `Mejor eficiencia_Colombia`, `Mejor eficiencia_internacional`) para obtener el total y sobre ese total se calcula el porcentaje de cada región. Regla clave: las variantes `LOW`/`MID`/`HIG` de una misma tecnología **comparten la misma distribución regional**.

El archivo final tiene **dos hojas**, ambas con códigos OSeMOSYS y formato `código | Anio | Antioquia..Suroccidente`:

1. **`Participacion_Fuel`** — participación regional por FUEL de OSeMOSYS (`INDCLIM`, `INDDHT`, ...). Cada FUEL agrupa todas las ramas de su Uso; `Otros` solo existe desglosado por combustible (`INDOTH_COA` = Carbón mineral, `INDOTH_ELC` = Electricidad) y sus demás combustibles quedan sin FUEL (se reportan y se excluyen).
2. **`Participacion_Technology`** — participación regional por TECHNOLOGY de OSeMOSYS (`DEMINDBAGBOI_LOW`, ...). Cada TECHNOLOGY se mapea a su familia LEAP `Uso\fuel` agregando las tres eficiencias, de modo que las variantes `LOW`/`MID`/`HIG` (y las de captura `CCS`) comparten distribución.

Reemplaza el cálculo de `Participacion_Regional_Industrial.ipynb` (que queda como referencia): aquel calculaba por `Uso\fuel\eficiencia` y su hoja `Participacion_Fuel` usaba los textos LEAP de Uso en vez de códigos.

Limpieza del insumo (igual que el notebook de referencia): reparación de rutas truncadas a 100 caracteres, se elimina el primer segmento `Subsector {Código}`, se omite el año 2021 y los totales nacionales nulos o cero producen participación 0.

In [1]:
import pandas as pd

pd.set_option('display.max_rows', 200)

In [2]:
# --- Configuración ---
INPUT_PATH = "Insumos/Datos_Industrial_Regionalizado.xlsx"
OUTPUT_PATH = "Insumos/Participacion_Regional_Industrial.xlsx"

HEADER_ROW = 5          # fila 0-indexada donde está el encabezado real ("Branch", 2021, 2022, ...)
ANIO_EXCLUIDO = 2021
COL_RUTA = "Branch"

# Mapeo a FUEL OSeMOSYS por (Uso, combustible LEAP) — 'Otros' es el único Uso que
# se desglosa por combustible (solo existen INDOTH_COA e INDOTH_ELC en el modelo)
MAPEO_USO_FUEL = {
    ("Otros", "Carbón mineral"): "INDOTH_COA",
    ("Otros", "Electricidad"): "INDOTH_ELC",
}

# Mapeo a FUEL OSeMOSYS por Uso completo (primer segmento), cuando no aplica el anterior
MAPEO_USO = {
    "Aire acondicionado": "INDCLIM",
    "Calor directo": "INDDHT",
    "Calor indirecto": "INDIHT",
    "Fuerza motriz": "INDMPW",
    "Iluminacion": "INDILU",
    "Refrigeracion": "INDREF",
}

## 1. Lectura de todas las hojas (regiones)

In [3]:
hojas = pd.read_excel(INPUT_PATH, sheet_name=None, skiprows=HEADER_ROW)
print("Regiones encontradas:", list(hojas.keys()))

primera = next(iter(hojas.values()))
print("Columnas de ejemplo:", primera.columns.tolist())

Regiones encontradas: ['Antioquia', 'Caribe', 'Este', 'Insular', 'Nordeste', 'Oriente', 'Suroccidente']
Columnas de ejemplo: ['Branch', 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054, 'Total']


## 2. Limpieza de la ruta y consolidación en formato largo

1. **Reparación de truncamiento**: el exporte de origen limita la ruta a 100 caracteres; se restaura el segmento final comparándolo contra los nombres canónicos observados.
2. Se elimina el primer segmento (`Subsector {Código}`), dejando la ruta como `Uso\fuel\eficiencia`, y se agrupan (sumando) las rutas duplicadas: los subsectores colapsan sobre la misma ruta.
3. Se descartan las filas `Total`, la columna `2021` y cualquier columna auxiliar.
4. Todo se consolida en un DataFrame maestro largo `Region | Tecnologia | Anio | Valor`.

In [4]:
def construir_mapa_reparacion(hojas: dict) -> dict:
    """Mapa {segmento_final_truncado: nombre_canonico_completo}.

    El origen limita la ruta a 100 caracteres y puede cortar el último
    segmento (ej. 'Mejor eficiencia_internaciona'). Un segmento es canónico
    si NO es prefijo estricto de otro segmento observado; cada segmento
    truncado se mapea al único canónico que lo contiene como prefijo.
    """
    finales = set()
    for df in hojas.values():
        finales |= {str(r).split("\\")[-1] for r in df[COL_RUTA].dropna()}

    canonicos = [s for s in finales if not any(o != s and o.startswith(s) for o in finales)]

    mapa = {}
    for seg in finales:
        candidatos = [c for c in canonicos if c.startswith(seg)]
        if len(candidatos) == 1:
            mapa[seg] = candidatos[0]
    return mapa


MAPA_REPARACION = construir_mapa_reparacion(hojas)
reparados = {k: v for k, v in MAPA_REPARACION.items() if k != v}
print(f"Segmentos truncados reparados: {len(reparados)}")
for trunc, completo in sorted(reparados.items()):
    print(f"  {trunc!r} -> {completo!r}")


def limpiar_ruta(ruta: str) -> str:
    """Repara el segmento final truncado y quita el primer segmento
    'Subsector {Codigo}'. Solo usa separación por backslash, sin cortes
    por longitud ni índices fijos de caracteres."""
    partes = str(ruta).split("\\")
    partes[-1] = MAPA_REPARACION.get(partes[-1], partes[-1])
    return "\\".join(partes[1:]) if len(partes) > 1 else partes[0]


def procesar_hoja(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Descartar filas sin ruta o filas de totales (ej. Branch == 'Total')
    df = df[df[COL_RUTA].notna()]
    df = df[df[COL_RUTA].astype(str).str.split("\\").str.len() >= 2]

    # Columnas de año: numéricas y dentro del rango válido (excluye 2021 y cualquier 'Total')
    year_cols = [
        c for c in df.columns
        if isinstance(c, (int, float)) and not pd.isna(c) and int(c) != ANIO_EXCLUIDO
    ]

    df["Tecnologia"] = df[COL_RUTA].apply(limpiar_ruta)

    df = df[["Tecnologia"] + year_cols]
    df.columns = ["Tecnologia"] + [int(c) for c in year_cols]

    # Agrupar rutas duplicadas tras la limpieza, sumando valores por año
    df = df.groupby("Tecnologia", as_index=False).sum(numeric_only=True)
    return df


def a_formato_largo(region: str, df: pd.DataFrame) -> pd.DataFrame:
    year_cols = [c for c in df.columns if c != "Tecnologia"]
    largo = df.melt(id_vars="Tecnologia", value_vars=year_cols, var_name="Anio", value_name="Valor")
    largo.insert(0, "Region", region)
    return largo


df_maestro = pd.concat(
    [a_formato_largo(region, procesar_hoja(df)) for region, df in hojas.items()],
    ignore_index=True,
)
df_maestro["Anio"] = df_maestro["Anio"].astype(int)
df_maestro["Valor"] = pd.to_numeric(df_maestro["Valor"], errors="coerce").fillna(0)

# Verificación: ninguna ruta debe quedar con segmento final no canónico (truncado)
CANONICOS_EFICIENCIA = {"Eficiencia_existente", "Mejor eficiencia_Colombia", "Mejor eficiencia_internacional"}
sospechosas = sorted({t for t in df_maestro["Tecnologia"].unique()
                      if t.split("\\")[-1] not in CANONICOS_EFICIENCIA})
assert not sospechosas, f"Rutas con segmento final no canónico: {sospechosas[:10]}"

print(df_maestro.shape)
df_maestro.head()

Segmentos truncados reparados: 4
  'Mejor eficiencia_Colombi' -> 'Mejor eficiencia_Colombia'
  'Mejor eficiencia_interna' -> 'Mejor eficiencia_internacional'
  'Mejor eficiencia_internacio' -> 'Mejor eficiencia_internacional'
  'Mejor eficiencia_internaciona' -> 'Mejor eficiencia_internacional'


(40194, 4)


,Region,Tecnologia,Anio,Valor
0,Antioquia,Aire acondicionado\Auto_cogeneración\Eficienci...,2022,0.000000
1,Antioquia,Aire acondicionado\Auto_cogeneración\Mejor efi...,2022,0.000000
2,Antioquia,Aire acondicionado\Auto_cogeneración\Mejor efi...,2022,0.000000
3,Antioquia,Aire acondicionado\Electricidad\Eficiencia_exi...,2022,0.084961
4,Antioquia,Aire acondicionado\Electricidad\Mejor eficienc...,2022,0.000585


## 3. Participación regional por FUEL de OSeMOSYS (hoja `Participacion_Fuel`)

Cada ruta `Uso\fuel\eficiencia` se mapea a un **FUEL de OSeMOSYS** según su Uso:

| FUEL | Corresponde a |
|---|---|
| `INDCLIM` | `Aire acondicionado` (todos los combustibles) |
| `INDDHT`  | `Calor directo` (todos los combustibles) |
| `INDIHT`  | `Calor indirecto` (todos los combustibles) |
| `INDILU`  | `Iluminacion` |
| `INDMPW`  | `Fuerza motriz` |
| `INDREF`  | `Refrigeracion` |
| `INDOTH_COA` | `Otros\Carbón mineral` |
| `INDOTH_ELC` | `Otros\Electricidad` |

El mapeo intenta primero por `(Uso, combustible)` (`MAPEO_USO_FUEL`, el desglose de `Otros`) y si no aplica, por el `Uso` completo (`MAPEO_USO`). Los combustibles de `Otros` distintos de carbón mineral y electricidad no tienen FUEL OSeMOSYS: se reportan y se excluyen del cálculo (cualquier ruta sin mapeo fuera de `Otros` detiene el notebook con error).

Se suman los valores por `Region`, `Fuel` y `Anio` (con lo que las eficiencias quedan agregadas), se calcula el total nacional por FUEL y la participación de cada región. Total nacional 0 (o nulo) produce participación 0.

In [5]:
def mapear_fuel_osemosys(tecnologia: str):
    """Devuelve el FUEL OSeMOSYS (ej. 'INDDHT') para una ruta
    'Uso\\fuel\\eficiencia', o None si no hay mapeo."""
    partes = tecnologia.split("\\")
    uso = partes[0]
    fuel_leap = partes[1] if len(partes) >= 2 else None
    return MAPEO_USO_FUEL.get((uso, fuel_leap)) or MAPEO_USO.get(uso)


df_fuel = df_maestro.copy()
df_fuel["Fuel"] = df_fuel["Tecnologia"].map(mapear_fuel_osemosys)

# Rutas sin FUEL OSeMOSYS: solo se admiten dentro de 'Otros' (no existe INDOTH_* para
# esos combustibles); cualquier otro Uso sin mapeo es un error de configuración.
rutas_sin_fuel = sorted(df_fuel.loc[df_fuel["Fuel"].isna(), "Tecnologia"].unique())
usos_sin_fuel = {r.split("\\")[0] for r in rutas_sin_fuel}
print(f"Rutas sin FUEL OSeMOSYS (excluidas del cálculo): {len(rutas_sin_fuel)}")
for f in sorted({"\\".join(r.split("\\")[:2]) for r in rutas_sin_fuel}):
    print(f"    {f}")
assert usos_sin_fuel <= {"Otros"}, f"Usos sin mapeo distintos de 'Otros': {sorted(usos_sin_fuel)}"

df_fuel = df_fuel[df_fuel["Fuel"].notna()]
print("FUELs OSeMOSYS encontrados:", sorted(df_fuel["Fuel"].unique()))

# Agrupar por Region, Fuel OSeMOSYS y Anio (suma todas las ramas y eficiencias del FUEL)
df_fuel = df_fuel.groupby(["Region", "Fuel", "Anio"], as_index=False)["Valor"].sum()

# Total nacional a nivel de FUEL y participación regional
df_total_fuel = (
    df_fuel.groupby(["Fuel", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)

df_participacion_fuel = pd.merge(df_fuel, df_total_fuel, on=["Fuel", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion_fuel["Total_Nacional"] = df_participacion_fuel["Total_Nacional"].fillna(0)
df_participacion_fuel["Participacion"] = 0.0
mask_valido = df_participacion_fuel["Total_Nacional"] != 0
df_participacion_fuel.loc[mask_valido, "Participacion"] = (
    df_participacion_fuel.loc[mask_valido, "Valor"]
    / df_participacion_fuel.loc[mask_valido, "Total_Nacional"]
)

# Salida por FUEL: regiones como columnas, indexado por Fuel y Año
df_salida_fuel = df_participacion_fuel.pivot_table(
    index=["Fuel", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida_fuel.columns.name = None
df_salida_fuel = df_salida_fuel.reset_index()

# Verificación: la suma de participaciones por Fuel-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols_fuel = [c for c in df_salida_fuel.columns if c not in ("Fuel", "Anio")]
suma_check_fuel = df_salida_fuel[region_cols_fuel].sum(axis=1)
filas_invalidas_fuel = ((suma_check_fuel - 1).abs() > 1e-6) & (suma_check_fuel.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas_fuel.sum()} de {len(df_salida_fuel)}")
df_salida_fuel.head(20)

Rutas sin FUEL OSeMOSYS (excluidas del cálculo): 42
    Otros\Auto_cogeneración
    Otros\Bagazo
    Otros\Biogas
    Otros\Carbón leña
    Otros\Coque
    Otros\Diesel
    Otros\Fuel oil
    Otros\GLP
    Otros\Gas natural
    Otros\Gasolina
    Otros\Jet fuel
    Otros\Leña
    Otros\Petróleo
    Otros\Resíduos
FUELs OSeMOSYS encontrados: ['INDCLIM', 'INDDHT', 'INDIHT', 'INDILU', 'INDMPW', 'INDOTH_COA', 'INDOTH_ELC', 'INDREF']


Filas con suma de participación distinta de 1 (y != 0): 0 de 264


,Fuel,Anio,Antioquia,Caribe,Este,Insular,Nordeste,Oriente,Suroccidente
0,INDCLIM,2022,0.099140,0.127357,0.000035,0.0,0.388403,0.192186,0.192879
1,INDCLIM,2023,0.095858,0.120218,0.000085,0.0,0.406224,0.189887,0.187729
2,INDCLIM,2024,0.095222,0.118567,0.000081,0.0,0.409294,0.188000,0.188836
3,INDCLIM,2025,0.096600,0.114574,0.000077,0.0,0.410134,0.188835,0.189780
4,INDCLIM,2026,0.096386,0.114731,0.000078,0.0,0.411463,0.187969,0.189372
5,INDCLIM,2027,0.096154,0.114983,0.000077,0.0,0.412573,0.187304,0.188910
6,INDCLIM,2028,0.096012,0.115179,0.000077,0.0,0.413146,0.187015,0.188571
7,INDCLIM,2029,0.095955,0.115440,0.000077,0.0,0.413232,0.186892,0.188404
8,INDCLIM,2030,0.095829,0.115720,0.000076,0.0,0.413758,0.186607,0.188010
9,INDCLIM,2031,0.095640,0.115851,0.000076,0.0,0.415649,0.186427,0.186357


## 4. Participación regional por TECHNOLOGY de OSeMOSYS (hoja `Participacion_Technology`)

Cada TECHNOLOGY del SAND se mapea a las rutas LEAP de su **familia tecnológica** `Uso\fuel` **agregando las tres eficiencias** (`Eficiencia_existente`, `Mejor eficiencia_Colombia`, `Mejor eficiencia_internacional`): se suma la demanda de la familia completa antes de sacar porcentajes, así que las variantes `LOW`/`MID`/`HIG` de una misma tecnología comparten la misma distribución regional.

- **Nomenclatura**: la familia se deriva del propio código, `DEMIND{FUEL}{USO}[CCS][_LOW|_MID|_HIG]`. Usos: `BOI` = Calor indirecto, `FUR` = Calor directo, `AIR` = Aire acondicionado, `ILU` = Iluminacion, `MPW` = Fuerza motriz, `OTH` = Otros, `REF` = Refrigeracion. Combustibles: `AUT` = Auto_cogeneración, `BAG` = Bagazo, `BGS` = Biogas, `COA` = Carbón mineral, `DSL` = Diesel, `ELC` = Electricidad, `HDG` = Hidrógeno, `LPG` = GLP, `NGS` = Gas natural, `WAS` = Resíduos.
- Las variantes de captura (`DEMINDBAGFURCCS`, `DEMINDNGSFURCSS`) comparten la distribución de su familia base.
- `DEMINDAUTBOI` / `DEMINDAUTFUR` no tienen sufijo de eficiencia: toman igualmente la familia agregada.
- Las ramas LEAP sin TECHNOLOGY en el SAND (Coque, Fuel oil, Gasolina, Jet fuel, Leña, Carbón leña, Petróleo, la mayoría de `Otros`, y Auto_cogeneración fuera de calor) se reportan al ejecutar y no generan filas.

In [6]:
# Los 70 DEMIND* del set TECHNOLOGY nacional (CSV_Nacional/TECHNOLOGY.csv)
TECNOLOGIAS_OSEMOSYS = """
DEMINDAUTBOI DEMINDAUTFUR
DEMINDBAGBOI_HIG DEMINDBAGBOI_LOW DEMINDBAGBOI_MID DEMINDBAGFURCCS DEMINDBAGFUR_HIG DEMINDBAGFUR_LOW DEMINDBAGFUR_MID
DEMINDBGSBOI_HIG DEMINDBGSBOI_LOW DEMINDBGSBOI_MID DEMINDBGSFUR_HIG DEMINDBGSFUR_LOW DEMINDBGSFUR_MID
DEMINDCOABOI_HIG DEMINDCOABOI_LOW DEMINDCOABOI_MID DEMINDCOAFUR_HIG DEMINDCOAFUR_LOW DEMINDCOAFUR_MID DEMINDCOAOTH_LOW
DEMINDDSLBOI_HIG DEMINDDSLBOI_LOW DEMINDDSLBOI_MID DEMINDDSLFUR_HIG DEMINDDSLFUR_LOW DEMINDDSLFUR_MID
DEMINDELCAIR_HIG DEMINDELCAIR_LOW DEMINDELCAIR_MID DEMINDELCBOI_HIG DEMINDELCBOI_LOW DEMINDELCBOI_MID
DEMINDELCFUR_HIG DEMINDELCFUR_LOW DEMINDELCFUR_MID DEMINDELCILU_HIG DEMINDELCILU_LOW DEMINDELCILU_MID
DEMINDELCMPW_HIG DEMINDELCMPW_LOW DEMINDELCMPW_MID DEMINDELCOTH_HIG DEMINDELCOTH_LOW DEMINDELCOTH_MID
DEMINDELCREF_HIG DEMINDELCREF_LOW DEMINDELCREF_MID
DEMINDHDGBOI_HIG DEMINDHDGBOI_LOW
DEMINDLPGBOI_HIG DEMINDLPGBOI_LOW DEMINDLPGBOI_MID DEMINDLPGFUR_HIG DEMINDLPGFUR_LOW DEMINDLPGFUR_MID
DEMINDNGSBOI_HIG DEMINDNGSBOI_LOW DEMINDNGSBOI_MID DEMINDNGSFURCSS DEMINDNGSFUR_HIG DEMINDNGSFUR_LOW DEMINDNGSFUR_MID
DEMINDWASBOI_HIG DEMINDWASBOI_LOW DEMINDWASBOI_MID DEMINDWASFUR_HIG DEMINDWASFUR_LOW DEMINDWASFUR_MID
""".split()

EFICIENCIA = {
    "LOW": "Eficiencia_existente",
    "MID": "Mejor eficiencia_Colombia",
    "HIG": "Mejor eficiencia_internacional",
}

# Códigos embebidos en el nombre: DEMIND{FUEL}{USO}[CCS|CSS][_LOW|_MID|_HIG]
CODIGO_USO = {
    "AIR": "Aire acondicionado",
    "BOI": "Calor indirecto",
    "FUR": "Calor directo",
    "ILU": "Iluminacion",
    "MPW": "Fuerza motriz",
    "OTH": "Otros",
    "REF": "Refrigeracion",
}
CODIGO_FUEL = {
    "AUT": "Auto_cogeneración",
    "BAG": "Bagazo",
    "BGS": "Biogas",
    "COA": "Carbón mineral",
    "DSL": "Diesel",
    "ELC": "Electricidad",
    "HDG": "Hidrógeno",
    "LPG": "GLP",
    "NGS": "Gas natural",
    "WAS": "Resíduos",
}


def familia_leap(tech: str):
    """'DEMINDBAGFUR_LOW' / 'DEMINDBAGFURCCS' / 'DEMINDAUTBOI' -> 'Calor directo\\Bagazo'.
    Devuelve None si el nombre no encaja en la convención DEMIND{FUEL}{USO}."""
    base = tech.removeprefix("DEMIND").split("_")[0]
    for sufijo in ("CCS", "CSS"):   # variantes de captura: comparten la familia base
        base = base.removesuffix(sufijo)
    fuel = CODIGO_FUEL.get(base[:3])
    uso = CODIGO_USO.get(base[3:])
    return f"{uso}\\{fuel}" if uso and fuel else None


# La participación es independiente de la eficiencia: cada TECHNOLOGY recibe las TRES
# ramas de eficiencia de su familia Uso\fuel (se suma toda la demanda antes de sacar %).
MAPEO_TECHNOLOGY, tec_sin_mapeo = {}, []
for tech in TECNOLOGIAS_OSEMOSYS:
    familia = familia_leap(tech)
    if familia is None:
        tec_sin_mapeo.append(tech)
    else:
        MAPEO_TECHNOLOGY[tech] = [f"{familia}\\{seg}" for seg in EFICIENCIA.values()]

print(f"TECHNOLOGYs del SAND: {len(TECNOLOGIAS_OSEMOSYS)}")
print(f"  Mapeadas a familia LEAP: {len(MAPEO_TECHNOLOGY)}")
print(f"  Sin familia LEAP (SIN_MAPEO): {len(tec_sin_mapeo)}")
for t in tec_sin_mapeo:
    print(f"    {t}")
assert not tec_sin_mapeo, "Hay TECHNOLOGYs que no encajan en la convención de nombres"

# Validación cruzada contra el insumo
rutas_insumo = set(df_maestro["Tecnologia"].unique())
rutas_mapeadas = {r for rutas in MAPEO_TECHNOLOGY.values() for r in rutas}
rutas_inexistentes = sorted(rutas_mapeadas - rutas_insumo)
assert not rutas_inexistentes, f"Rutas mapeadas que NO existen en el insumo: {rutas_inexistentes}"

rutas_sin_technology = sorted(rutas_insumo - rutas_mapeadas)
familias_sin_technology = sorted({"\\".join(r.split("\\")[:2]) for r in rutas_sin_technology})
print(f"Ramas LEAP sin TECHNOLOGY asociada: {len(rutas_sin_technology)} rutas en {len(familias_sin_technology)} familias Uso\\fuel:")
for f in familias_sin_technology:
    print(f"    {f}")

TECHNOLOGYs del SAND: 70
  Mapeadas a familia LEAP: 70
  Sin familia LEAP (SIN_MAPEO): 0
Ramas LEAP sin TECHNOLOGY asociada: 99 rutas en 33 familias Uso\fuel:
    Aire acondicionado\Auto_cogeneración
    Calor directo\Carbón leña
    Calor directo\Coque
    Calor directo\Fuel oil
    Calor directo\Gasolina
    Calor directo\Hidrógeno
    Calor directo\Jet fuel
    Calor directo\Leña
    Calor directo\Petróleo
    Calor indirecto\Carbón leña
    Calor indirecto\Coque
    Calor indirecto\Fuel oil
    Calor indirecto\Gasolina
    Calor indirecto\Jet fuel
    Calor indirecto\Leña
    Calor indirecto\Petróleo
    Fuerza motriz\Auto_cogeneración
    Iluminacion\Auto_cogeneración
    Otros\Auto_cogeneración
    Otros\Bagazo
    Otros\Biogas
    Otros\Carbón leña
    Otros\Coque
    Otros\Diesel
    Otros\Fuel oil
    Otros\GLP
    Otros\Gas natural
    Otros\Gasolina
    Otros\Jet fuel
    Otros\Leña
    Otros\Petróleo
    Otros\Resíduos
    Refrigeracion\Auto_cogeneración


In [7]:
# Valores por TECHNOLOGY: se suman las rutas LEAP asociadas antes de calcular la participación
df_map_tech = pd.DataFrame(
    [(tech, ruta) for tech, rutas in MAPEO_TECHNOLOGY.items() for ruta in rutas],
    columns=["Technology", "Tecnologia"],
)

df_tech = df_map_tech.merge(df_maestro, on="Tecnologia", how="left")
df_tech = df_tech.groupby(["Region", "Technology", "Anio"], as_index=False)["Valor"].sum()

# Total nacional por TECHNOLOGY y participación regional
df_total_tech = (
    df_tech.groupby(["Technology", "Anio"], as_index=False)["Valor"]
    .sum()
    .rename(columns={"Valor": "Total_Nacional"})
)

df_participacion_tech = pd.merge(df_tech, df_total_tech, on=["Technology", "Anio"], how="left")

# Manejo de división por cero / nulos: si el total nacional es 0 (o NaN), la participación es 0
df_participacion_tech["Total_Nacional"] = df_participacion_tech["Total_Nacional"].fillna(0)
df_participacion_tech["Participacion"] = 0.0
mask_valido = df_participacion_tech["Total_Nacional"] != 0
df_participacion_tech.loc[mask_valido, "Participacion"] = (
    df_participacion_tech.loc[mask_valido, "Valor"]
    / df_participacion_tech.loc[mask_valido, "Total_Nacional"]
)

# Salida por TECHNOLOGY: regiones como columnas, indexado por Technology y Año
df_salida_tech = df_participacion_tech.pivot_table(
    index=["Technology", "Anio"],
    columns="Region",
    values="Participacion",
    fill_value=0,
)
df_salida_tech.columns.name = None
df_salida_tech = df_salida_tech.reset_index()

# Verificación: la suma de participaciones por Technology-Anio debe ser ~1 (o 0 si el total nacional es 0)
region_cols_tech = [c for c in df_salida_tech.columns if c not in ("Technology", "Anio")]
suma_check_tech = df_salida_tech[region_cols_tech].sum(axis=1)
filas_invalidas_tech = ((suma_check_tech - 1).abs() > 1e-6) & (suma_check_tech.abs() > 1e-6)
print(f"Filas con suma de participación distinta de 1 (y != 0): {filas_invalidas_tech.sum()} de {len(df_salida_tech)}")
df_salida_tech.head(20)

Filas con suma de participación distinta de 1 (y != 0): 0 de 2310


,Technology,Anio,Antioquia,Caribe,Este,Insular,Nordeste,Oriente,Suroccidente
0,DEMINDAUTBOI,2022,0.024836,0.034142,0.000219,0.0,0.043058,0.072216,0.825529
1,DEMINDAUTBOI,2023,0.049796,0.024385,0.000119,0.0,0.031999,0.080786,0.812916
2,DEMINDAUTBOI,2024,0.052159,0.024182,0.000120,0.0,0.031764,0.080700,0.811076
3,DEMINDAUTBOI,2025,0.051386,0.023510,0.000116,0.0,0.031577,0.079886,0.813524
4,DEMINDAUTBOI,2026,0.051367,0.023585,0.000117,0.0,0.031727,0.079704,0.813500
5,DEMINDAUTBOI,2027,0.051347,0.023684,0.000117,0.0,0.031866,0.079618,0.813367
6,DEMINDAUTBOI,2028,0.051340,0.023756,0.000117,0.0,0.031943,0.079636,0.813207
7,DEMINDAUTBOI,2029,0.051337,0.023822,0.000116,0.0,0.031957,0.079659,0.813109
8,DEMINDAUTBOI,2030,0.051351,0.023917,0.000116,0.0,0.032040,0.079694,0.812883
9,DEMINDAUTBOI,2031,0.051609,0.024111,0.000115,0.0,0.032403,0.080206,0.811557


## 5. Exportar resultado

Un solo archivo Excel con **dos hojas**: `Participacion_Fuel` y `Participacion_Technology`, ambas en formato pivote (`código | Anio | una columna por región`).

In [8]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df_salida_fuel.to_excel(writer, sheet_name="Participacion_Fuel", index=False)
    df_salida_tech.to_excel(writer, sheet_name="Participacion_Technology", index=False)

print(f"Archivo exportado en: {OUTPUT_PATH}")

# Los 8 FUELs OSeMOSYS esperados están presentes
fuels_esperados = set(MAPEO_USO.values()) | set(MAPEO_USO_FUEL.values())
fuels_salida = set(df_salida_fuel["Fuel"].unique())
faltantes = fuels_esperados - fuels_salida
print(f"FUELs en la salida: {len(fuels_salida)} de {len(fuels_esperados)} esperados")
assert not faltantes, f"FUELs faltantes: {sorted(faltantes)}"

techs_salida = set(df_salida_tech["Technology"].unique())
print(f"TECHNOLOGYs con participación: {len(techs_salida)} de {len(MAPEO_TECHNOLOGY)} mapeadas")
assert techs_salida == set(MAPEO_TECHNOLOGY), "Hay TECHNOLOGYs mapeadas sin participación calculada"

# Regla clave: las variantes de una misma familia comparten distribución regional
_familias = {}
for t in sorted(techs_salida):
    _familias.setdefault(familia_leap(t), []).append(t)
for fam, ts in _familias.items():
    base = df_salida_tech[df_salida_tech["Technology"] == ts[0]].drop(columns="Technology").reset_index(drop=True)
    for otro in ts[1:]:
        alt = df_salida_tech[df_salida_tech["Technology"] == otro].drop(columns="Technology").reset_index(drop=True)
        assert base.equals(alt), f"{ts[0]} y {otro} no comparten distribución"
print(f"Verificado: las {len(techs_salida)} TECHNOLOGYs comparten distribución dentro de sus {len(_familias)} familias")

Archivo exportado en: Insumos/Participacion_Regional_Industrial.xlsx
FUELs en la salida: 8 de 8 esperados
TECHNOLOGYs con participación: 70 de 70 mapeadas


Verificado: las 70 TECHNOLOGYs comparten distribución dentro de sus 25 familias
